Explanation

Daily grain: Each row is a single metric for a single day.

Customer growth: (new_customers / previous_day_total_customers) * 100 → null-safe.

Repeat customer rate: % of customers with >1 order in last 30 days.

Top 10 products: Calculates top 10 product sales as a % of total sales per day.

Profit margin & avg_discount: Derived directly from mart_sales_daily.

Long format: Perfect for dashboards — BI tools can pivot or filter easily.

In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os, sys

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")

from superstore_platform_config import get_catalog, get_kpi_schema, get_mart_schema, get_semantic_layer_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
mart_schema = get_mart_schema()
semantic_layer_schema = get_semantic_layer_schema()  # e.g., dev_platform

from superstore_logger import get_superstore_logger, log_event
from superstore_backfill_utils import is_dry_run
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_metrics_business_kpi = get_superstore_logger("superstore_metrics_business_kpi")


# -----------------------------
# Dry run
# -----------------------------
# The serving layer rebuilds from Gold and has no run-mode logic of its own —
# a replay or backfill simply produces different Gold for it to read. But it
# does write, so it must not write during a dry run. is_dry_run reads only the
# dry_run parameter and never raises, so a serving task cannot fail on the
# full_refresh gate that is enforced upstream where it belongs.
if is_dry_run(dbutils):
    log_event(
        logger_metrics_business_kpi,
        "INFO",
        "Dry run: skipping metrics_business_kpi write",
        dry_run=True,
        layer="kpi view"
    )
    dbutils.notebook.exit("DRY_RUN_COMPLETED layer=metrics_business_kpi (no data written)")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_metrics_business_kpi,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="kpi view"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
# Check if the required tables exist before creating the feature table
# The tables the view below actually reads. This used to name facts_orders --
# a Gold table, never present in the mart schema, and not read by the view --
# so the check always failed and the view was never created in any environment.
required_tables = [
    f"{catalog}.{mart_schema}.mart_sales_daily"
]

missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, mart_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_metrics_business_kpi, "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create metrics business kpi view.", layer="kpi view")
    # Raise, do not log-and-continue: a logged ERROR on a task that exits
    # normally reports SUCCESS, which is how a missing view went unnoticed.
    raise RuntimeError(f"Missing tables: {', '.join(missing_tables)}. Cannot create metrics business kpi view.")
else:
    # -------------------------------
    # Create kpi View (Semantic Layer for Analysts)
    # -------------------------------
    spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.{semantic_layer_schema}.metrics_business_kpi AS
    -- Step 1: Daily Sales Metrics
    WITH daily_sales AS (
        SELECT
            order_date AS metric_date,
            SUM(total_sales) AS total_sales,
            SUM(total_profit) AS total_profit,
            CASE WHEN SUM(total_sales) = 0 THEN NULL ELSE ROUND(SUM(total_profit)/SUM(total_sales),4) END AS profit_margin,
            AVG(avg_discount) AS avg_discount
        FROM {catalog}.{mart_schema}.mart_sales_daily
        GROUP BY order_date
    ),

    -- Steps 2-3: customer metrics, defined relative to each metric_date and
    -- never to the current date. Both used to anchor on "today": on data that
    -- ends before today, repeat_customer_rate returned no rows at all and
    -- customer_growth_rate was 0 on every day ("new" meant "ordered today", not
    -- "ordered for the first time"). Defined this way they are correct for every
    -- day in the data, however old it is.

    -- One row per customer per day they ordered.
    customer_days AS (
        SELECT DISTINCT customer_id, order_date
        FROM {catalog}.{mart_schema}.mart_sales_daily
        WHERE customer_id IS NOT NULL AND order_date IS NOT NULL
    ),

    -- Step 2: customer growth = customers whose FIRST order was on metric_date,
    -- as a % of all customers who had ordered before it. Every order day is
    -- present, with 0 when nobody was new; the first day is NULL (no base).
    first_orders AS (
        SELECT customer_id, MIN(order_date) AS first_order_date
        FROM customer_days
        GROUP BY customer_id
    ),
    new_by_day AS (
        SELECT first_order_date AS metric_date, COUNT(*) AS new_customers
        FROM first_orders
        GROUP BY first_order_date
    ),
    order_days AS (
        SELECT DISTINCT order_date AS metric_date FROM customer_days
    ),
    customer_growth AS (
        SELECT
            d.metric_date,
            COALESCE(n.new_customers, 0) AS new_customers,
            SUM(COALESCE(n.new_customers, 0)) OVER (
                ORDER BY d.metric_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ) AS customers_before
        FROM order_days d
        LEFT JOIN new_by_day n ON n.metric_date = d.metric_date
    ),

    -- Step 3: repeat rate = of the customers who ordered on metric_date, the %
    -- who had also ordered in the 30 days before it. A rolling window per day,
    -- so it needs no "as of" date at all.
    with_previous AS (
        SELECT
            customer_id,
            order_date,
            LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
        FROM customer_days
    ),
    repeat_by_day AS (
        SELECT
            order_date AS metric_date,
            COUNT(*) AS customers,
            COUNT_IF(previous_order_date >= DATE_SUB(order_date, 30)) AS repeat_customers_30d
        FROM with_previous
        GROUP BY order_date
    ),

    -- Step 4: Top 10 Product Sales % per Day
    top_products AS (
        SELECT
            order_date AS metric_date,
            SUM(total_sales) AS total_sales,
            SUM(total_sales) FILTER (WHERE rank <= 10) AS top_10_sales
        FROM (
            SELECT
                order_date,
                product_id,
                total_sales,
                ROW_NUMBER() OVER (PARTITION BY order_date ORDER BY total_sales DESC) AS rank
            FROM {catalog}.{mart_schema}.mart_sales_daily
        ) ranked
        GROUP BY order_date
    )

    -- Step 5: Combine all metrics in long format
    SELECT metric_date, 'total_sales' AS metric_name, total_sales AS metric_value FROM daily_sales
    UNION ALL
    SELECT metric_date, 'total_profit', total_profit FROM daily_sales
    UNION ALL
    SELECT metric_date, 'profit_margin', profit_margin FROM daily_sales
    UNION ALL
    SELECT metric_date, 'avg_discount', avg_discount FROM daily_sales
    UNION ALL
    SELECT metric_date, 'customer_growth_rate' AS metric_name,
        CASE WHEN customers_before IS NULL OR customers_before = 0 THEN NULL
             ELSE ROUND(new_customers * 100.0 / customers_before, 2) END AS metric_value
    FROM customer_growth
    UNION ALL
    SELECT metric_date, 'repeat_customer_rate' AS metric_name,
        CASE WHEN customers = 0 THEN NULL
             ELSE ROUND(repeat_customers_30d * 100.0 / customers, 2) END AS metric_value
    FROM repeat_by_day
    UNION ALL
    SELECT tp.metric_date, 'top_10_product_sales_pct' AS metric_name,
        CASE WHEN tp.total_sales = 0 THEN NULL ELSE ROUND(tp.top_10_sales * 100.0 / tp.total_sales,2) END AS metric_value
    FROM top_products tp;
    """)
log_event(logger_metrics_business_kpi, "INFO", "Metrics business kpi view created successfully.", layer="kpi view")
